# 🧮 IDx Partners — Notebook Final: LGD Fix + Kalibrasi PD + ECL/IFRS9

Urutan pipeline:
**Load & definisi populasi → Split → Preprocessing (fit di train) → XGBoost (`scale_pos_weight`, tanpa SMOTE) → Kalibrasi PD (isotonic) → LGD fix → EAD → ECL → Staging IFRS9**


In [ ]:
!pip install xgboost -q

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, classification_report
from scipy.stats import ks_2samp
from xgboost import XGBClassifier
import joblib

pd.set_option('display.max_columns', None)

DATA_PATH = '/content/loan_data_2007_2014 (1).csv'   

## 1. Load Data & Definisi Populasi

Tiga populasi yang dipakai (agar tidak tertukar):
- **PD training** — loan yang sudah *resolved*: `Fully Paid`, `Charged Off`, `Default`, + 2 status "does not meet credit policy"
- **LGD eligible** — subset resolved yang benar-benar final (`out_prncp = 0`): HANYA `Charged Off` (+ does not meet policy: charged off). `Default` dikeluarkan karena `out_prncp`-nya masih hidup, belum final.
- **Live/scoring** — loan yang belum resolved: `Current`, `In Grace Period`, `Late (16-30 days)`, `Late (31-120 days)`. Di sini `PD × LGD × EAD` diterapkan buat ECL forward-looking.

In [ ]:
data_awal = pd.read_csv(DATA_PATH, index_col=0, low_memory=False)
print('Shape awal:', data_awal.shape)
print('Duplikat (raw, seluruh kolom):', data_awal.duplicated().sum())
print('id unique:', data_awal['id'].nunique(), '/', len(data_awal))

# --- kolom yang di-drop PERMANEN (tidak dibutuhkan di manapun) ---
drop_admin = ['id','member_id','url','desc','emp_title','title','zip_code',
              'policy_code','pymnt_plan','application_type',
              'funded_amnt','funded_amnt_inv']

drop_leakage_unused = ['out_prncp_inv','total_pymnt_inv','total_rec_int',
                        'total_rec_late_fee','last_pymnt_d','last_pymnt_amnt',
                        'next_pymnt_d','last_credit_pull_d']

drop_sparse = ['annual_inc_joint','dti_joint','verification_status_joint',
               'open_acc_6m','open_il_6m','open_il_12m','open_il_24m',
               'mths_since_rcnt_il','total_bal_il','il_util','open_rv_12m',
               'open_rv_24m','max_bal_bc','all_util','inq_fi','total_cu_tl',
               'inq_last_12m']  # 100% missing di vintage 2007-2014

drop_all = drop_admin + drop_leakage_unused + drop_sparse
data = data_awal.drop(columns=[c for c in drop_all if c in data_awal.columns])
print('Shape setelah drop kolom admin/leakage-unused/sparse:', data.shape)

# kolom ini TETAP disimpan (dipakai LGD/EAD/staging), TIDAK boleh masuk fitur PD:
# loan_status, out_prncp, total_rec_prncp, recoveries, collection_recovery_fee, total_pymnt

Shape awal: (466285, 74)
Duplikat (raw, seluruh kolom): 0
id unique: 466285 / 466285
Shape setelah drop kolom admin/leakage-unused/sparse: (466285, 37)


In [ ]:
completed_status = [
    'Fully Paid',
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off'
]

live_status = ['Current', 'In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)']

# ---- populasi PD training ----
data_completed = data[data['loan_status'].isin(completed_status)].copy()

def define_default(status):
    if status in ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']:
        return 1
    return 0

data_completed['default_flag'] = data_completed['loan_status'].apply(define_default)

# flag kelayakan LGD -- HANYA loan yang out_prncp sudah 0 (final)
data_completed['lgd_eligible'] = data_completed['loan_status'].isin(
    ['Charged Off', 'Does not meet the credit policy. Status:Charged Off']
)

# ---- populasi live/scoring ----
data_live = data[data['loan_status'].isin(live_status)].copy()

print(f"Populasi PD training : {len(data_completed):,} baris | default rate: {data_completed['default_flag'].mean():.2%}")
print(f"Populasi LGD eligible: {data_completed['lgd_eligible'].sum():,} baris")
print(f"Populasi live/scoring: {len(data_live):,} baris")
print()
print(data_completed['loan_status'].value_counts())
print()
print(data_live['loan_status'].value_counts())

Populasi PD training : 230,795 baris | default rate: 19.09%
Populasi LGD eligible: 43,236 baris
Populasi live/scoring: 235,490 baris

loan_status
Fully Paid                                             184739
Charged Off                                             42475
Does not meet the credit policy. Status:Fully Paid       1988
Default                                                   832
Does not meet the credit policy. Status:Charged Off       761
Name: count, dtype: int64

loan_status
Current               224226
Late (31-120 days)      6900
In Grace Period         3146
Late (16-30 days)       1218
Name: count, dtype: int64


## 2. Split (leakage-free)

`feature_cols` dipakai sebagai **allowlist** (bukan drop-list) — `loan_status`, `out_prncp`, `total_rec_prncp`, `recoveries`, `collection_recovery_fee`, `total_pymnt` sengaja TIDAK masuk `X` karena itu kolom post-outcome (persis penyebab bug AUC=1.0 di draft sebelumnya).

In [ ]:
feature_cols = ['loan_amnt','term','int_rate','installment','grade','sub_grade',
                 'emp_length','home_ownership','annual_inc','verification_status',
                 'purpose','addr_state','dti','delinq_2yrs','inq_last_6mths',
                 'open_acc','pub_rec','revol_bal','revol_util','total_acc',
                 'initial_list_status','collections_12_mths_ex_med','acc_now_delinq',
                 'tot_coll_amt','tot_cur_bal','total_rev_hi_lim']

lgd_ead_cols = ['loan_status','out_prncp','total_rec_prncp','recoveries',
                'collection_recovery_fee','total_pymnt','lgd_eligible']

target_col = 'default_flag'

X   = data_completed[feature_cols].copy()
y   = data_completed[target_col].copy()
aux = data_completed[lgd_ead_cols].copy()

X_train, X_test, y_train, y_test, aux_train, aux_test = train_test_split(
    X, y, aux, test_size=0.2, stratify=y, random_state=42
)

assert (X_train.index == aux_train.index).all()
assert (X_test.index == aux_test.index).all()

print(f"Train: {X_train.shape[0]:,} ({y_train.mean():.2%} default)")
print(f"Test : {X_test.shape[0]:,} ({y_test.mean():.2%} default)")

Train: 184,636 (19.09% default)
Test : 46,159 (19.09% default)


## 3. Preprocessing (fit hanya di train)

`emp_length` & `term` dibersihkan dulu (deterministik, aman diterapkan kapan saja — bukan statistik yang bisa bocor). Imputer/encoder/scaler di-`fit()` di `X_train`, lalu dipakai buat `.transform()` ke `X_test` **dan** populasi live nanti — bukan di-fit ulang.

In [ ]:
def clean_emp_term(df):
    df = df.copy()
    df['emp_length_missing'] = df['emp_length'].isnull().astype(int)
    df['emp_length'] = (
        df['emp_length'].astype(str)
        .str.replace(' years?', '', regex=True)
        .str.replace('< 1', '0', regex=False)
        .str.replace(r'10\+', '10', regex=True)
    )
    df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce').fillna(-1)

    df['term'] = df['term'].astype(str).str.strip().str.replace(' months', '', regex=False)
    df['term'] = pd.to_numeric(df['term'], errors='coerce')
    return df

X_train = clean_emp_term(X_train)
X_test  = clean_emp_term(X_test)

numeric_features     = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
print(f"Numerik ({len(numeric_features)}): {numeric_features}")
print(f"Kategorikal ({len(categorical_features)}): {categorical_features}")

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

preprocessor.fit(X_train)   # FIT DI TRAIN SAJA

X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)
print(f"\nX_train: {X_train.shape} -> {X_train_processed.shape}")
print(f"X_test : {X_test.shape} -> {X_test_processed.shape}")

Numerik (20): ['loan_amnt', 'term', 'int_rate', 'installment', 'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'emp_length_missing']
Kategorikal (7): ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status']

X_train: (184636, 27) -> (184636, 137)
X_test : (46159, 27) -> (46159, 137)


## 4. Model — XGBoost (tanpa SMOTE)

Karena XGBoost tidak punya parameter `class_weight` langsung seperti sklearn estimators — padanannya `scale_pos_weight` (rasio kelas negatif/positif). Fungsinya sama: bobot lebih besar ke kelas minoritas (default) tanpa membuat data sintetis.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_processed, y_train)

y_proba_raw = xgb_model.predict_proba(X_test_processed)[:, 1]
y_pred_raw  = (y_proba_raw >= 0.5).astype(int)

ks_raw = ks_2samp(y_proba_raw[y_test==1], y_proba_raw[y_test==0]).statistic
print(f"\n📊 XGBoost (belum dikalibrasi) — evaluasi di TEST:")
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba_raw):.4f}")
print(f"KS      : {ks_raw:.4f}")
print(classification_report(y_test, y_pred_raw, target_names=['Good','Default']))

scale_pos_weight: 4.237

📊 XGBoost (belum dikalibrasi) — evaluasi di TEST:
ROC-AUC : 0.7145
KS      : 0.3144
              precision    recall  f1-score   support

        Good       0.89      0.65      0.75     37345
     Default       0.31      0.66      0.42      8814

    accuracy                           0.65     46159
   macro avg       0.60      0.66      0.59     46159
weighted avg       0.78      0.65      0.69     46159



## 5. Kalibrasi PD (`CalibratedClassifierCV`, isotonic)

Wajib sebelum ECL — PD mentah dari model tree-based biasanya bagus buat *ranking* tapi angka probabilitasnya sendiri bisa bias. `cv=5` supaya kalibrasi di-fit di fold terpisah dari data yang dipakai fit model dasarnya (tidak overfit ke data yang sama).

In [ ]:
base_model_for_calib = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc', random_state=42, n_jobs=-1
)

calibrated_model = CalibratedClassifierCV(base_model_for_calib, method='isotonic', cv=5)
calibrated_model.fit(X_train_processed, y_train)

y_proba_cal = calibrated_model.predict_proba(X_test_processed)[:, 1]
y_pred_cal  = (y_proba_cal >= 0.5).astype(int)

ks_cal = ks_2samp(y_proba_cal[y_test==1], y_proba_cal[y_test==0]).statistic
print(f"📊 XGBoost (SUDAH dikalibrasi) — evaluasi di TEST:")
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba_cal):.4f}")
print(f"KS      : {ks_cal:.4f}")
print(f"Brier score sebelum kalibrasi: {brier_score_loss(y_test, y_proba_raw):.4f}")
print(f"Brier score sesudah kalibrasi: {brier_score_loss(y_test, y_proba_cal):.4f}  (lebih rendah = lebih baik)")
print(classification_report(y_test, y_pred_cal, target_names=['Good','Default']))

📊 XGBoost (SUDAH dikalibrasi) — evaluasi di TEST:
ROC-AUC : 0.7142
KS      : 0.3130
Brier score sebelum kalibrasi: 0.2145
Brier score sesudah kalibrasi: 0.1398  (lebih rendah = lebih baik)
              precision    recall  f1-score   support

        Good       0.82      0.99      0.90     37345
     Default       0.58      0.06      0.11      8814

    accuracy                           0.81     46159
   macro avg       0.70      0.52      0.50     46159
weighted avg       0.77      0.81      0.74     46159



## 6. LGD — Formula Fix

Formula lama salah: `1 − recoveries/loan_amnt` (tidak kredit `total_rec_prncp`, tetapi menggunakan `loan_amnt` mentah) → LGD ekstrem 94%+.
Formula benar: **`LGD = 1 − (total_rec_prncp + recoveries) / loan_amnt`**, dihitung HANYA dari populasi `lgd_eligible` di **TRAIN**, dikalibrasi per grade.

In [ ]:
lgd_base_train = aux_train.copy()
lgd_base_train['loan_amnt'] = X_train['loan_amnt']
lgd_base_train['grade'] = X_train['grade']

lgd_train_pop = lgd_base_train[lgd_base_train['lgd_eligible']].copy()

before_clip = 1 - ((lgd_train_pop['total_rec_prncp'] + lgd_train_pop['recoveries']) / lgd_train_pop['loan_amnt'])
lgd_train_pop['lgd_row'] = before_clip.clip(0, 1)

n_clip_low  = (before_clip < 0).sum()
n_clip_high = (before_clip > 1).sum()
print(f"Populasi LGD eligible (train): {len(lgd_train_pop):,} baris")
print(f"Clip < 0: {n_clip_low} ({n_clip_low/len(lgd_train_pop):.2%}) | Clip > 1: {n_clip_high}")
print(lgd_train_pop['lgd_row'].describe())

lgd_stats_by_grade = lgd_train_pop.groupby('grade')['lgd_row'].agg(['mean','count'])
lgd_by_grade = lgd_stats_by_grade['mean']
lgd_overall  = lgd_train_pop['lgd_row'].mean()

print("\nLGD per grade (train):")
print(lgd_stats_by_grade)
print(f"\nLGD overall (fallback): {lgd_overall:.2%}")

Populasi LGD eligible (train): 34,585 baris
Clip < 0: 132 (0.38%) | Clip > 1: 0
count    34585.000000
mean         0.675248
std          0.202038
min          0.000000
25%          0.564847
50%          0.719276
75%          0.822045
max          1.000000
Name: lgd_row, dtype: float64

LGD per grade (train):
           mean  count
grade                 
A      0.574572   2011
B      0.609957   7408
C      0.668673   9613
D      0.695285   7968
E      0.745530   4710
F      0.761432   2236
G      0.778436    639

LGD overall (fallback): 67.52%


## 7. EAD & Scoring Populasi Live

EAD untuk populasi live = `out_prncp` (outstanding balance saat ini). PD diambil dari model terkalibrasi, fitur di-*transform* mengunakan `preprocessor` yang sudah di-fit di train (**bukan** fit ulang).

In [ ]:
X_live = data_live[feature_cols].copy()
X_live = clean_emp_term(X_live)
X_live_processed = preprocessor.transform(X_live)   # transform saja, TIDAK fit ulang

pd_lifetime = calibrated_model.predict_proba(X_live_processed)[:, 1]

# PD 12 bulan -- approksimasi dari PD lifetime pakai constant hazard rate,
# karena model ini nggak eksplisit horizon-nya (limitasi didokumentasikan di bagian 9)
term_months = X_live['term'].values.astype(float)
term_months = np.where(term_months <= 0, 36, term_months)  # safety fallback
monthly_hazard = 1 - (1 - pd_lifetime) ** (1 / term_months)
pd_12m = 1 - (1 - monthly_hazard) ** 12
pd_12m = np.clip(pd_12m, 0, 1)

results_live = data_live[['loan_status', 'out_prncp']].copy()
results_live['grade'] = X_live['grade'].values
results_live['pd_lifetime'] = pd_lifetime
results_live['pd_12m'] = pd_12m
results_live['LGD'] = results_live['grade'].map(lgd_by_grade).fillna(lgd_overall)
results_live['EAD'] = results_live['out_prncp']

print(results_live[['pd_lifetime','pd_12m','LGD','EAD']].describe())

         pd_lifetime         pd_12m            LGD            EAD
count  235490.000000  235490.000000  235490.000000  235490.000000
mean        0.211780       0.064151       0.651707    8695.000827
std         0.130379       0.039194       0.054097    6504.268053
min         0.000000       0.000000       0.574572       0.000000
25%         0.109307       0.034192       0.609957    3431.280000
50%         0.184703       0.056283       0.668673    7192.730000
75%         0.298554       0.086829       0.695285   12425.132500
max         0.888889       0.355606       0.778436   31898.770000


## 8. Staging IFRS9 (Stage 1–3) & ECL

- **Stage 1** (`Current`) → ECL 12 bulan
- **Stage 2** (`In Grace Period`, `Late (16-30 days)`) → ECL lifetime
- **Stage 3** (`Late (31-120 days)`) → ECL lifetime — **catatan:** bucket ini menyilang batas DPD 90 (SICR vs default), dataset nggak punya DPD granular, jadi ditempatkan konservatif di Stage 3 (didokumentasikan sebagai limitasi, bukan disembunyikan).

In [ ]:
def assign_stage(status):
    if status == 'Current':
        return 1
    elif status in ['In Grace Period', 'Late (16-30 days)']:
        return 2
    elif status == 'Late (31-120 days)':
        return 3
    return np.nan

results_live['stage'] = results_live['loan_status'].apply(assign_stage)

results_live['ECL'] = np.where(
    results_live['stage'] == 1,
    results_live['pd_12m'] * results_live['LGD'] * results_live['EAD'],
    results_live['pd_lifetime'] * results_live['LGD'] * results_live['EAD']
)

summary_stage = results_live.groupby('stage').agg(
    n_loans=('EAD', 'count'),
    total_EAD=('EAD', 'sum'),
    avg_PD=('pd_lifetime', 'mean'),
    avg_LGD=('LGD', 'mean'),
    total_ECL=('ECL', 'sum')
).round(2)
summary_stage['pct_portfolio'] = (summary_stage['n_loans'] / summary_stage['n_loans'].sum() * 100).round(1)
summary_stage['coverage_ratio'] = (summary_stage['total_ECL'] / summary_stage['total_EAD']).round(4)

print("📋 Ringkasan ECL per Stage IFRS9:")
print(summary_stage)
print(f"\nTotal ECL portofolio: {results_live['ECL'].sum():,.2f}")
print(f"Total EAD portofolio: {results_live['EAD'].sum():,.2f}")
print(f"Coverage ratio keseluruhan: {results_live['ECL'].sum()/results_live['EAD'].sum():.2%}")

📋 Ringkasan ECL per Stage IFRS9:
       n_loans     total_EAD  avg_PD  avg_LGD    total_ECL  pct_portfolio  \
stage                                                                       
1       224226  1.937235e+09    0.21     0.65  86060951.92           95.2   
2         4364  4.105260e+07    0.26     0.68   8568831.72            1.9   
3         6900  6.929841e+07    0.27     0.68  14666840.37            2.9   

       coverage_ratio  
stage                  
1              0.0444  
2              0.2087  
3              0.2116  

Total ECL portofolio: 109,296,624.01
Total EAD portofolio: 2,047,585,744.75
Coverage ratio keseluruhan: 5.34%


## 9. Limitasi & Simpan Output

**Limitasi yang perlu didokumentasikan (transparan, bukan disembunyikan):**
- Model bersifat **TTC (through-the-cycle)**, bukan PIT — tanpa overlay makroekonomi (GDP growth, unemployment).
- LGD flat per-segmen (grade), bukan model regresi LGD terpisah — cukup buat untuk proyek ini, model LGD penuh adalah paket lanjutan.
- PD 12-bulan adalah **approksimasi** dari PD lifetime pakai constant hazard rate, bukan model survival/panel terpisah — karena dataset ini nggak punya struktur outcome bulanan.
- `Late (31-120 days)` digabung jadi Stage 3 secara konservatif karena keterbatasan granularitas DPD di dataset.

In [ ]:
joblib.dump({
    'preprocessor': preprocessor,
    'calibrated_model': calibrated_model,
    'lgd_by_grade': lgd_by_grade.to_dict(),
    'lgd_overall': lgd_overall,
    'feature_cols': feature_cols,
    'results_live': results_live,
    'summary_stage': summary_stage,
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
}, 'ifrs9_final_output.pkl')

results_live.to_csv('ifrs9_scoring_results.csv', index=False)
summary_stage.to_csv('ifrs9_stage_summary.csv')

print("✅ Notebook selesai. Output tersimpan: ifrs9_final_output.pkl, ifrs9_scoring_results.csv, ifrs9_stage_summary.csv")
print("Angka-angka di 'summary_stage' ini yang dipakai buat isi placeholder di app.py Streamlit & insight storytelling.")

✅ Notebook selesai. Output tersimpan: ifrs9_final_output.pkl, ifrs9_scoring_results.csv, ifrs9_stage_summary.csv
Angka-angka di 'summary_stage' ini yang dipakai buat isi placeholder di app.py Streamlit & insight storytelling.
